# PathSnap: Distributed Map-Matching & Route Reconstruction
## Phase 1 — Dataset Exploration, Quality Audit, Preprocessing & Folium Visualization

**Course:** PES University UE24CS352A Machine Learning  
**Project:** PathSnap — Distributed Map-Matching and Route Reconstruction  
**Phase:** 1 (Dataset Exploration & Foundations)  

---

### Phase 1 Objectives & Guidelines:
1. **Dataset Understanding:** Comprehensively inspect the raw dataset structure across permitted partitions (`train`: 80 trips, `student_test`: 10 trips).
2. **Zero Leakage:** Strictly preserve partition boundaries. Do not touch or access evaluator-testing data.
3. **Integrity & Indexing Verification:** Confirm 0-based node and arc indexing, column types, coordinate validity, and timestamp ordering.
4. **Data Quality Audit:** Identify missing coordinates, duplicate readings, irregular sampling intervals, GPS dropouts/gaps, and velocity spikes.
5. **Preprocessing & Geometric Utilities:** Construct road-segment geometries and compute geodesic/Haversine distances without modifying raw data.
6. **Folium Geospatial Visualization:** Render interactive maps displaying raw GPS points, road network geometry, and ground truth routes with clear legends and controls.


In [ ]:
# 1. Imports and Workspace Configuration
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium

# Ensure project root is in python path
project_root = os.path.abspath("..")
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from src.data_loader import load_trip, list_trips, verify_trip_integrity, resolve_partition_dir
from src.preprocessing import (
    haversine_distance,
    audit_and_preprocess_trajectory,
    construct_road_segment_geometries,
    point_to_segment_distance
)
from src.visualization import generate_trip_map
from src.eda import run_full_eda, generate_eda_plots

# Visual settings
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({"font.size": 11, "figure.autolayout": True})

print("Environment initialized successfully.")


## 2. Dataset Partition Inventory & File Availability

The dataset is divided into:
- **Training partition (`train`):** 80 trips for training, development, and cross-validation.
- **Student testing partition (`student_test`):** 10 trips for development-stage evaluation and model comparison.
- **Evaluator testing partition:** 10 trips reserved exclusively for final evaluation (strictly excluded from our pipeline).

Each trip directory contains 4 file types:
1. `.track`: GPS observations `[longitude, latitude, timestamp]` (whitespace-separated, no header).
2. `.nodes`: Network nodes `[longitude, latitude]` (0-based row index represents `node_id`).
3. `.arcs`: Network segments `[from_node_id, to_node_id]` (0-based row index represents `arc_id`).
4. `.route`: Ground-truth chronological sequence of traversed `arc_id`s.


In [ ]:
# 2. Inspect partition contents and trip counts
train_trips = list_trips("train", base_dir="..")
test_trips = list_trips("student_test", base_dir="..")

print(f"Number of training trips:     {len(train_trips)}")
print(f"Number of student-test trips: {len(test_trips)}")
print(f"Total permitted trips:        {len(train_trips) + len(test_trips)}")

# Verify file presence for a sample trip
sample_id = train_trips[0]
sample_trip = load_trip(sample_id, partition="train", base_dir="..")
print(f"\nLoaded Sample Trip: {sample_trip}")


## 3. Dataset Summary Tables & Comparative Statistics

We load the precomputed `results/dataset_summary.csv` containing metrics for all 90 permitted trips:
- Number of GPS observations
- Number of road nodes and arcs
- Reference route arc counts
- Physical trajectory length (metres) and duration (seconds)
- File sizes on disk


In [ ]:
# 3. Load and inspect dataset_summary.csv
summary_csv = os.path.join(project_root, "results", "dataset_summary.csv")
summary_df = pd.read_csv(summary_csv)

# Aggregate statistics grouped by partition
partition_stats = summary_df.groupby("partition").agg({
    "trip_id": "count",
    "track_points": ["sum", "min", "mean", "max"],
    "nodes_count": ["min", "mean", "max"],
    "arcs_count": ["min", "mean", "max"],
    "route_arcs_count": ["min", "mean", "max"],
    "trajectory_length_m": ["min", "mean", "max"],
    "route_continuity_pct": ["min", "mean"]
}).round(2)

display(partition_stats)


## 4. Comprehensive Data Quality Audit

To guarantee robustness before building map-matching models, we inspect:
1. **Missing or invalid coordinates:** Latitude outside $[-90, 90]$ or Longitude outside $[-180, 180]$.
2. **Timestamp monotonicity & ordering:** Any negative time increments $\Delta t < 0$.
3. **Sampling interval distribution:** Frequency of nominal 1.0s sampling vs irregular intervals.
4. **GPS dropouts & gaps:** Sequences where $\Delta t > 5.0\text{s}$.
5. **Duplicate readings:** Stationary observations where coordinates repeat across consecutive seconds.
6. **Velocity spikes:** Anomalous displacement indicating GPS jumps ($v > 180\text{ km/h}$).
7. **Graph reference integrity:** Verifying that all `from_node_id` and `to_node_id` values lie within $[0, N_{nodes}-1]$ and route `arc_id` values lie within $[0, N_{arcs}-1]$.


In [ ]:
# 4. Load and inspect data_quality_report.csv
quality_csv = os.path.join(project_root, "results", "data_quality_report.csv")
quality_df = pd.read_csv(quality_csv)

quality_summary = pd.DataFrame({
    "Total Trips Audited": [len(quality_df)],
    "Total GPS Observations": [summary_df["track_points"].sum()],
    "Total Missing Coordinates": [quality_df["missing_coords"].sum()],
    "Total Missing Timestamps": [quality_df["missing_timestamps"].sum()],
    "Invalid Coordinate Values": [quality_df["invalid_coords"].sum()],
    "Invalid Node References in Arcs": [quality_df["invalid_node_references"].sum()],
    "Invalid Arc References in Route": [quality_df["invalid_arc_references"].sum()],
    "Non-monotonic Timestamps": [quality_df["non_monotonic_timestamps"].sum()],
    "Duplicate Coordinates (Stationary)": [quality_df["duplicate_coords"].sum()],
    "Large GPS Dropouts (>5s)": [quality_df["large_gaps_count_gt_5s"].sum()],
    "Velocity Spikes (>180 km/h)": [quality_df["speed_anomalies_gt_180kmh"].sum()],
    "Mean % of Nominal 1.0s Sampling": [f"{quality_df['sampling_1s_pct'].mean():.2f}%"]
}).T.rename(columns={0: "Audit Metric"})

display(quality_summary)


## 5. Exploratory Data Distributions & Visual Analysis

Below we visualize the empirical distributions across training and student-testing partitions:
1. Trajectory length and observation counts.
2. Sampling frequency stability and dropout gaps.
3. Road-network scale and topological disconnected components.
4. Route continuity and relationship between GPS observations and route arcs.


In [ ]:
# 5. Display EDA Visualizations
fig_dir = os.path.join(project_root, "results", "figures")

fig_files = [
    "eda_trajectory_distributions.png",
    "eda_sampling_and_gaps.png",
    "eda_road_network_complexity.png",
    "eda_route_structure.png"
]

for fname in fig_files:
    fpath = os.path.join(fig_dir, fname)
    if os.path.exists(fpath):
        plt.figure(figsize=(12, 5))
        img = plt.imread(fpath)
        plt.imshow(img)
        plt.axis("off")
        plt.title(fname.replace("eda_", "").replace(".png", "").replace("_", " ").title(), fontsize=13, pad=10)
        plt.show()


## 6. Preprocessing & Road Geometry Construction

We demonstrate the reusable preprocessing functions:
1. Constructing road segment geometries from `.nodes` and `.arcs` (calculating segment lengths, bearings, and midpoints).
2. Auditing a trajectory and tagging anomalies without altering the raw underlying data.
3. Calculating point-to-segment perpendicular distance in metres using equirectangular projection and Haversine distance.


In [ ]:
# 6. Demonstrate Preprocessing & Geometric Utilities on Sample Trip
trip = load_trip("00000001", partition="train", base_dir="..")

# Construct segment geometries
segments_df = construct_road_segment_geometries(trip.nodes, trip.arcs)
print("Constructed Road Segments Sample:")
display(segments_df[["from_node_id", "to_node_id", "from_lon", "from_lat", "to_lon", "to_lat", "length_m", "azimuth_deg"]].head())

# Preprocess and audit trajectory
annotated_track, trip_audit = audit_and_preprocess_trajectory(trip.track)
print(f"\nTrajectory Preprocessing Audit for Trip {trip.trip_id}:")
print(f"- Total Points: {trip_audit['n_observations']}")
print(f"- Trajectory Physical Length: {trip_audit['total_length_metres']/1000.0:.2f} km")
print(f"- Max Speed Recorded: {trip_audit['speed_max_kmh']:.1f} km/h (Mean: {trip_audit['speed_mean_kmh']:.1f} km/h)")
print(f"- Large Gaps (>5s): {trip_audit['gap_count']} (Max Gap: {trip_audit['gap_max_sec']:.1f}s)")


## 7. Interactive Folium Geospatial Map

We generate and inspect an interactive Folium map for training trip `00000001`:
- **Blue Points / Dashed Line:** Raw GPS observations with popups indicating observation index, timestamp, and coordinates.
- **Slate Gray Lines:** Road network segments within the trajectory corridor.
- **Bold Green Line:** Reference ground-truth route sequence.
- **Start / End Markers:** Distinct icons marking trajectory departure and destination.
- **Interactive Controls:** Layer control toggle and custom floating map legend.

The standalone interactive map is saved to `maps/sample_trip_map.html`.


In [ ]:
# 7. Generate Folium Map for Sample Trip
map_path = os.path.join(project_root, "maps", "sample_trip_map.html")
trip_map = generate_trip_map(
    trip=trip,
    output_html_path=map_path,
    corridor_buffer_m=300.0,
    show_gps_dots=True
)

print(f"Map successfully generated and saved to: {map_path}")
print(f"HTML File Size: {os.path.getsize(map_path):,} bytes")


## 8. Summary of Findings & Next Steps (Phase 2 Readiness)

### Key EDA Takeaways:
1. **Coordinate & Indexing Validity:**
   - 100% of node references and arc references are strictly valid 0-based indices.
   - 0 missing coordinates, 0 missing timestamps, 0 coordinate range violations.
2. **Sampling Dynamics:**
   - Nominally 1.0 Hz with an overall average of 94.3% exact 1-second intervals.
   - Significant GPS dropouts (gaps up to 160 seconds) exist across 1,260 instances. Downstream HMM transition probability models must dynamically adjust for $\Delta t$ and $\Delta d$.
3. **Trajectory vs Route Cardinality:**
   - GPS observation counts do **not** equal reference route arc counts ($N_{track} \neq N_{route}$). Map matching is not a trivial 1-to-1 row correspondence; a vehicle spends variable time on individual road arcs.
4. **Road Network Scale:**
   - Graph sizes range from small urban corridors (649 arcs) to large regional graphs (307,392 arcs). Candidate selection with spatial indexing (KD-Trees / R-Trees) will be essential for real-time inference under 5-second deadlines.
